# CNN v2 · from pixels to a prediction

This notebook explores the **released** MobileNetV3 happy/sad face-crop model.
Use the environment from `requirements.txt` and install `jupyterlab` / `ipykernel`
if your editor does not already provide a notebook kernel.

The model scored **91.4% on 1,469 audited FER face crops**. It expects a closely
cropped face, not a full scene. It labels visual expression, not internal feelings.
See `MODEL_CARD.md` for the full-photo regression and other limitations.


In [ ]:
from pathlib import Path
import json
import os
import torch
from IPython.display import display, Image as NotebookImage

root = Path.cwd()
if not (root / 'cnn_v2').exists() and (root.parent / 'cnn_v2').exists():
    root = root.parent
assert (root / 'cnn_v2').exists(), 'Start the notebook in the cnn repository.'
os.chdir(root)
torch.set_num_threads(4)
from cnn_v2.predict import Predictor
predictor = Predictor()
print('Classes:', predictor.metadata['class_names'])
print('Parameters:', f"{predictor.metadata['parameters']:,}")
print('Validation-selected epoch:', predictor.metadata['best_epoch'])


## 1. Inspect the evidence

The original CNN and v2 were evaluated on exactly the same FER test IDs.
Training and validation images are excluded from that evaluation. The comparison
changes both the architecture and training data; it is not an architecture ablation.


In [ ]:
benchmark = json.loads((root / 'artifacts/v2/benchmark.json').read_text())
for name in ['original_model_same_test', 'majority_baseline', 'new_model']:
    metrics = benchmark[name]
    print(f"{name:30} accuracy={metrics['accuracy']:.2%}  AUC={metrics['roc_auc']:.3f}")
display(NotebookImage(filename=str(root / 'docs/figures/benchmark.png')))


## 2. Understand the input and output

1. Correct orientation and convert the crop to grayscale.
2. Resize to 128×128 and replicate into three channels.
3. Normalize each channel using ImageNet statistics.
4. Extract features with MobileNetV3 and score two classes with the new head.
5. Divide logits by the validation-fitted temperature, then apply softmax.

The larger score wins. The 0.65 low-confidence marker is a UI aid, not a detector
for non-faces or unfamiliar images.


In [ ]:
# Change this path to a closely cropped photo of one face.
image_path = root / 'my_face_crop.jpg'
if image_path.exists():
    display(NotebookImage(filename=str(image_path), width=220))
    print(json.dumps(predictor.predict(image_path), indent=2))
else:
    print('Set image_path to your own face crop, then run this cell again.')


## 3. Reproduce training

Run these commands in a terminal using this notebook's Python environment:

```bash
python -m cnn_v2.dataset
python -m cnn_v2.train --output runs/notebook-experiment --epochs 20
python -m cnn_v2.evaluate --run runs/notebook-experiment
```

The first command downloads the pinned dataset and audits duplicate clusters.
The second uses training and validation data only. The third evaluates the chosen
checkpoint on test images. Always use a new run directory.

Suggested learning exercise: use a new training run to compare frozen features
with full fine-tuning **using validation data**. Do not repeatedly tune against the
published test set. Reserve a new external holdout for subsequent decisions.

For the original TensorFlow walkthrough, use
`notebooks/LegacyImageClassification.ipynb` in a separate Python 3.12 environment
installed from `requirements-legacy.txt`.
